# Fase 3: modelado predictivo para StreamView

Introducción: esta fase convierte la evidencia del EDA y del feature engineering en un flujo de modelado real. El objetivo es entrenar un modelo que estime el potencial de éxito de cada contenido y deje una base reproducible para validación posterior, priorización y dashboard.

Conclusión: la fase 3 cierra la parte predictiva inicial con separación train/test, preprocessing, benchmarking de modelos y guardado de artefactos reutilizables para la siguiente etapa del proyecto.


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score, mean_absolute_error, mean_squared_error, precision_score, r2_score, recall_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import joblib

project_root = Path.cwd().resolve()
if not (project_root / 'data' / 'processed').exists():
    for parent in project_root.parents:
        if (parent / 'data' / 'processed').exists():
            project_root = parent
            break

processed_dir = project_root / 'data' / 'processed'
raw_path = processed_dir / 'catalogo_streamview.csv'
if not raw_path.exists():
    raise FileNotFoundError(f'Falta el dataset base en {raw_path}.')

catalogo = pd.read_csv(raw_path, low_memory=False)
print(f'Filas cargadas: {len(catalogo):,}')
print(f'Columnas: {catalogo.columns.tolist()[:12]} ...')


Filas cargadas: 32,000
Columnas: ['type', 'title', 'director', 'cast', 'country', 'date_added', 'release_year', 'rating', 'genres', 'language', 'popularity', 'vote_count'] ...


In [2]:
for column in ['popularity', 'vote_count', 'vote_average', 'budget', 'revenue', 'release_year']:
    catalogo[column] = pd.to_numeric(catalogo[column], errors='coerce')

catalogo = catalogo.drop_duplicates(subset=['title', 'type', 'release_year'], keep='first').copy()
print('Duplicados eliminados:', catalogo.shape[0])
print(catalogo[['type', 'title', 'release_year', 'popularity', 'vote_count', 'vote_average', 'budget', 'revenue']].head().to_string(index=False))
print('\nResumen de nulos:')
print(catalogo.isna().sum().sort_values(ascending=False).head(10).to_string())


Duplicados eliminados: 31918
 type                                        title  release_year  popularity  vote_count  vote_average      budget     revenue
Movie                          Shrek Forever After          2010     203.893        7449         6.380 165000000.0 752600867.0
Movie                                    Inception          2010     156.242       37119         8.369 160000000.0 839030630.0
Movie Harry Potter and the Deathly Hallows: Part 1          2010     121.191       19327         7.744 250000000.0 954305868.0
Movie                                      Tangled          2010     111.762       11638         7.600 260000000.0 592461732.0
Movie                     How to Train Your Dragon          2010     110.044       13259         7.800 165000000.0 494879471.0

Resumen de nulos:
budget          15955
revenue         15955
director        11066
country          2256
cast             1357
genres           1076
title               0
release_year        0
date_added    

In [3]:
def min_max(series):
    series = pd.to_numeric(series, errors='coerce')
    min_value = series.min()
    max_value = series.max()
    if pd.isna(min_value) or pd.isna(max_value) or max_value == min_value:
        return pd.Series(0.0, index=series.index)
    return (series - min_value) / (max_value - min_value)

catalogo['vote_count_log'] = np.log1p(catalogo['vote_count'].fillna(0).clip(lower=0))
catalogo['budget_log'] = np.log1p(catalogo['budget'].fillna(0).clip(lower=0))
catalogo['revenue_log'] = np.log1p(catalogo['revenue'].fillna(0).clip(lower=0))

catalogo['popularity_norm'] = min_max(catalogo['popularity'].fillna(catalogo['popularity'].median()))
catalogo['vote_count_norm'] = min_max(catalogo['vote_count_log'])
catalogo['vote_average_norm'] = min_max(catalogo['vote_average'].fillna(catalogo['vote_average'].median()))

catalogo['indice_exito'] = (
    0.40 * catalogo['popularity_norm'] +
    0.30 * catalogo['vote_count_norm'] +
    0.30 * catalogo['vote_average_norm']
).clip(0, 1)

catalogo['profit'] = catalogo['revenue'] - catalogo['budget']
catalogo['roi_approx'] = np.where(catalogo['budget'] > 0, catalogo['profit'] / catalogo['budget'], np.nan)
roi_valid = catalogo['roi_approx'].notna()
if roi_valid.any():
    catalogo.loc[roi_valid, 'roi_norm'] = min_max(catalogo.loc[roi_valid, 'roi_approx'])
    catalogo['roi_norm'] = catalogo['roi_norm'].fillna(0.0)
else:
    catalogo['roi_norm'] = 0.0

movie_mask = catalogo['type'].eq('Movie')
if movie_mask.any():
    catalogo.loc[movie_mask, 'indice_exito_financiero'] = (
        0.80 * catalogo.loc[movie_mask, 'indice_exito'] +
        0.20 * catalogo.loc[movie_mask, 'roi_norm']
    ).clip(0, 1)
else:
    catalogo['indice_exito_financiero'] = catalogo['indice_exito']

catalogo['soporte_valoracion'] = catalogo['vote_count'].ge(100).astype('int8')
low_cut, high_cut = catalogo['indice_exito'].quantile([0.33, 0.66]).tolist()
labels = ['Bajo Exito', 'Medio Exito', 'Alto Exito']
catalogo['categoria_exito'] = pd.cut(
    catalogo['indice_exito'],
    bins=[-np.inf, low_cut, high_cut, np.inf],
    labels=labels,
    right=False,
).astype(str)
print('Cortes del índice de éxito:', round(low_cut, 4), 'y', round(high_cut, 4))
print(catalogo['categoria_exito'].value_counts().to_string())


Cortes del índice de éxito: 0.2679 y 0.3366
categoria_exito
Alto Exito     10852
Medio Exito    10533
Bajo Exito     10533


In [4]:
feature_columns = [
    'type', 'title', 'country', 'genres', 'language', 'rating',
    'release_year', 'popularity', 'vote_count', 'vote_average',
    'budget', 'revenue', 'soporte_valoracion', 'vote_count_log'
]
feature_columns = [column for column in feature_columns if column in catalogo.columns]

X = catalogo[feature_columns].copy()
y_reg = catalogo['indice_exito'].astype(float)
y_clf = catalogo['categoria_exito']

X_train, X_valid, y_reg_train, y_reg_valid = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)
_, _, y_clf_train, y_clf_valid = train_test_split(
    X, y_clf, test_size=0.2, random_state=42
)

numeric_cols = X.select_dtypes(include=['number']).columns.tolist()
categorical_cols = [column for column in X.columns if column not in numeric_cols]
print(f'Variables numéricas: {numeric_cols}')
print(f'Variables categóricas: {categorical_cols}')
print(f'Train size: {X_train.shape[0]} | Validation size: {X_valid.shape[0]}')


Variables numéricas: ['rating', 'release_year', 'popularity', 'vote_count', 'vote_average', 'budget', 'revenue', 'soporte_valoracion', 'vote_count_log']
Variables categóricas: ['type', 'title', 'country', 'genres', 'language']
Train size: 25534 | Validation size: 6384


In [6]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
            ]),
            numeric_cols,
        ),
        (
            'cat',
            Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
            ]),
            categorical_cols,
        ),
    ],
    remainder='drop',
)

base_regression = Pipeline([
    ('preprocess', preprocessor),
    ('model', LinearRegression()),
])
base_regression.fit(X_train, y_reg_train)
reg_pred = base_regression.predict(X_valid)
reg_mae = mean_absolute_error(y_reg_valid, reg_pred)
reg_mse = mean_squared_error(y_reg_valid, reg_pred)
reg_rmse = np.sqrt(reg_mse)
reg_r2 = r2_score(y_reg_valid, reg_pred)
print('Regresión lineal:')
print({'mae': round(reg_mae, 4), 'rmse': round(reg_rmse, 4), 'r2': round(reg_r2, 4)})

base_classifier = Pipeline([
    ('preprocess', preprocessor),
    ('model', RandomForestClassifier(random_state=42, n_estimators=300, class_weight='balanced_subsample')),
])
base_classifier.fit(X_train, y_clf_train)
clf_pred = base_classifier.predict(X_valid)
print('\nClasificación base:')
print({
    'accuracy': round(accuracy_score(y_clf_valid, clf_pred), 4),
    'precision_weighted': round(precision_score(y_clf_valid, clf_pred, average='weighted', zero_division=0), 4),
    'recall_weighted': round(recall_score(y_clf_valid, clf_pred, average='weighted', zero_division=0), 4),
    'f1_weighted': round(f1_score(y_clf_valid, clf_pred, average='weighted', zero_division=0), 4),
})
print(classification_report(y_clf_valid, clf_pred, zero_division=0))


Regresión lineal:
{'mae': 0.0, 'rmse': np.float64(0.0), 'r2': 1.0}

Clasificación base:
{'accuracy': 0.9591, 'precision_weighted': 0.9607, 'recall_weighted': 0.9591, 'f1_weighted': 0.9594}
              precision    recall  f1-score   support

  Alto Exito       0.98      0.95      0.97      2172
  Bajo Exito       0.99      0.96      0.97      2103
 Medio Exito       0.91      0.97      0.94      2109

    accuracy                           0.96      6384
   macro avg       0.96      0.96      0.96      6384
weighted avg       0.96      0.96      0.96      6384



In [ ]:
rf_reg_param_grid = {
    'model__n_estimators': [200],
    'model__max_depth': [None, 10],
    'model__min_samples_leaf': [1, 2],
}
rf_clf_param_grid = {
    'model__n_estimators': [200],
    'model__max_depth': [None, 10],
    'model__min_samples_leaf': [1, 2],
    'model__class_weight': ['balanced', 'balanced_subsample'],
}

rf_reg_pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', RandomForestRegressor(random_state=42)),
])
rf_clf_pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', RandomForestClassifier(random_state=42)),
])

grid_reg = GridSearchCV(rf_reg_pipeline, param_grid=rf_reg_param_grid, cv=2, n_jobs=1, scoring='neg_mean_absolute_error')
grid_clf = GridSearchCV(rf_clf_pipeline, param_grid=rf_clf_param_grid, cv=2, n_jobs=1, scoring='f1_weighted')

grid_reg.fit(X_train, y_reg_train)
grid_clf.fit(X_train, y_clf_train)

best_reg = grid_reg.best_estimator_
best_clf = grid_clf.best_estimator_

reg_pred_tuned = best_reg.predict(X_valid)
clf_pred_tuned = best_clf.predict(X_valid)

print('Mejor regresión:', grid_reg.best_params_)
print({
    'mae': round(mean_absolute_error(y_reg_valid, reg_pred_tuned), 4),
    'rmse': round(np.sqrt(mean_squared_error(y_reg_valid, reg_pred_tuned)), 4),
    'r2': round(r2_score(y_reg_valid, reg_pred_tuned), 4),
})
print('\nMejor clasificación:', grid_clf.best_params_)
print({
    'accuracy': round(accuracy_score(y_clf_valid, clf_pred_tuned), 4),
    'precision_weighted': round(precision_score(y_clf_valid, clf_pred_tuned, average='weighted', zero_division=0), 4),
    'recall_weighted': round(recall_score(y_clf_valid, clf_pred_tuned, average='weighted', zero_division=0), 4),
    'f1_weighted': round(f1_score(y_clf_valid, clf_pred_tuned, average='weighted', zero_division=0), 4),
})


In [ ]:
model_dir = project_root / 'models'
model_dir.mkdir(exist_ok=True)

joblib.dump(best_reg, model_dir / 'best_regression_model.pkl')
joblib.dump(best_clf, model_dir / 'best_classifier_model.pkl')

results = {
    'regression': {
        'mae': float(mean_absolute_error(y_reg_valid, reg_pred_tuned)),
        'rmse': float(np.sqrt(mean_squared_error(y_reg_valid, reg_pred_tuned))),
        'r2': float(r2_score(y_reg_valid, reg_pred_tuned)),
    },
    'classification': {
        'accuracy': float(accuracy_score(y_clf_valid, clf_pred_tuned)),
        'precision_weighted': float(precision_score(y_clf_valid, clf_pred_tuned, average='weighted', zero_division=0)),
        'recall_weighted': float(recall_score(y_clf_valid, clf_pred_tuned, average='weighted', zero_division=0)),
        'f1_weighted': float(f1_score(y_clf_valid, clf_pred_tuned, average='weighted', zero_division=0)),
    },
    'grid_reg_params': grid_reg.best_params_,
    'grid_clf_params': grid_clf.best_params_,
}

with (model_dir / 'phase3_model_results.json').open('w', encoding='utf-8') as results_file:
    json.dump(results, results_file, indent=2, ensure_ascii=False)

print(f'\nModelos guardados en: {model_dir}')
print(json.dumps(results, ensure_ascii=False, indent=2))
